## Bernstein Vazirani Algorithm
#### In this tutorial, we will demonstrate the capabilities of qBraid Algorithms Bernstein Vazirani Module
Begin by importing the module from qBraid Algorithms library

In [ ]:
%%capture

%pip install 'qbraid[visualization]' qbraid-algorithms

In [ ]:
import pyqasm
from qbraid_algorithms import bernstein_vazirani as bv

To load a full Bernstein Vazirani algorithm circuit as a PyQASM module, simply pass the secret string to be encoded in the oracle to the `load_program()` method.

In [ ]:
module = bv.generate_program("1011")

We can perform all standard [PyQASM](https://docs.qbraid.com/pyqasm/user-guide/overview) operations on the module, such as unrolling.

In [ ]:
module.unroll()

Below, we display the unrolled circuit, which includes preparing the input and ancilla qubits, applying the '1011' oracle, applying Hadamard gates after the oracle, then measuring the results.

In [ ]:
module_str = pyqasm.dumps(module)
print(module_str)

## Using B-V in your own OpenQASM3 program
#### qBraid algorithms makes it easy to incorporate either the full Bernstein Vazirani algorithm - or just the encoded oracle - into your own OpenQASM3 circuit.
To use a secret string-encoded oracle in your circuit, first generate the oracle submodule using the `generate_oracle` method, which takes a secret string. The method will create a QASM3 file containing your oracle as a subroutine within your current working directory.

In [ ]:
bv.generate_oracle("111")

Below, we can see the custom oracle subroutine that you now have access to. To use this in your own circuit, simply add `include "oracle.qasm";` to your OpenQASM file, and call the `oracle` subroutine by passing an appropriately sized register of qubits and an ancilla qubit.

In [ ]:
%cat oracle.qasm

Similarly, you can generate an entire Bernstein Vazirani circuit as a submodule using the `generate_subroutine` method, again passing your desired secret string.

In [ ]:
subroutine = bv.save_to_qasm("011")

To use the subroutine in your own circuit, add `include "bernvaz.qasm";` to your OpenQASM file, and call the `bernvaz` method by passing an appropriately sized register of qubits, as well as an ancilla qubit.

In [ ]:
%cat bernvaz.qasm

## Running Algorithms on qBraid
Running algorithms on qBraid is simple. First, import QbraidProvider from qBraid Runtime. Visit [here](https://docs.qbraid.com/sdk/user-guide/providers/native#qbraidprovider) for more information.

In [ ]:
from qbraid.runtime import QbraidProvider

If you have not yet configured QbraidProvider, provide your API key.

In [ ]:
# provider = QbraidProvider(api_key='API_KEY')
provider = QbraidProvider()

We'll run our program on qBraid's QIR simulator.

In [ ]:
device = provider.get_device("qbraid:qbraid:sim:qir-sv")

To run the job, simply pass a QASM string to the device. If you have a PyQASM module, use `dumps` to generate a QASM string

In [ ]:
secret_string = "101"
module = bv.generate_program(secret_string)
module.unroll()
qasm_str = pyqasm.dumps(module)
print(qasm_str)

In [ ]:
job = device.run(qasm_str, shots=500)

We can now get the counts from the job results.

In [ ]:
results = job.result()
counts = results.data.measurement_counts

We can now check the counts to see if the most frequent value is our secret string. This particular backend includes the ancilla qubit as the least significant bit, so we'll remove that.

In [ ]:
# Remove ancilla qubit from bitstring
processed_counts = {bitstr[:-1]: count for bitstr, count in counts.items()}
# Find most frequent count
max_str = max(processed_counts, key=processed_counts.get)

print(f"Secret String: {secret_string}. B-V result string: {max_str}")

We see that the algorithm successfully identified the secret string. Finally, we can plot the results using qBraid Visualization.

In [ ]:
from qbraid.visualization import plot_histogram

plot_histogram(counts)